# BODAQS One-step Suspension Metrics - Self-scoped

This notebook preprocesses an explicit batch into one configured library run, then opens a chart-scope selector and the simple suspension metrics dashboard for the freshly processed sessions.

In [1]:
from pathlib import Path
import sys

from IPython.display import display
import pandas as pd
import plotly.io as pio


def find_analysis_dir(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "bodaqs_analysis").is_dir():
            return candidate
        analysis = candidate / "analysis"
        if (analysis / "bodaqs_analysis").is_dir():
            return analysis
    raise RuntimeError("Could not find the BODAQS analysis package root from the current working directory.")


ANALYSIS_DIR = find_analysis_dir()
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

LIBRARIES_ROOT = Path.home() / "OneDrive" / "BODAQS-data"
LIBRARY_ID = "archie"

# Add CSV, ZIP, or BDQ paths here. Every requested input is processed.
INPUT_PATHS = [
    # Path(r"C:\\Users\\benco\\OneDrive\\BODAQS-data\\sources\\example\\inbox\\session.zip"),
]

PREPROCESS_PROFILE_PATH = ANALYSIS_DIR / "config" / "preprocess_profiles" / "suspension_default_v1.json"
BIKE_PROFILE_PATH = ANALYSIS_DIR / "config" / "bike_profiles" / "example_enduro_bike_v1.json"

ATTACH_DRAFT_NOTE = False
SESSION_NOTE_TEMPLATE_PATH = ANALYSIS_DIR / "templates" / "session_note_templates" / "suspension_setup" / "1.0.json"

GENERIC_LOG_METADATA_PATHS = [ANALYSIS_DIR / "config" / "log_metadata_examples"]
LOG_METADATA_PATH = None
FIT_DIR = Path.home() / "OneDrive" / "BODAQS-data" / "sources" / "ben-stevo-local" / "fit"
FIT_BINDINGS_PATH = ANALYSIS_DIR / "config" / "fit_bindings_v1.json"
LOGGER_TIMEZONE = None
RUN_TZ_LABEL = "AWST"
RUN_DESCRIPTION = "One-step suspension metrics batch"

FRONT_SUSPENSION_SELECTOR = {"end": "front", "domain": "wheel", "quantity": "disp", "unit": "mm"}
REAR_SUSPENSION_SELECTOR = {"end": "rear", "domain": "wheel", "quantity": "disp", "unit": "mm"}
FRONT_EVENT_SIGNAL_SELECTOR = {"end": "front", "domain": "wheel", "quantity": "disp"}
REAR_EVENT_SIGNAL_SELECTOR = {"end": "rear", "domain": "wheel", "quantity": "disp"}
SCATTER_COMPRESSION_EVENT_ID = "compressions_all>25"
SCATTER_REBOUND_EVENT_ID = "rebounds_all>25"
SCATTER_X_METRIC = "speed_mps"
SCATTER_COMPRESSION_Y_METRIC = "compression_mm"
SCATTER_REBOUND_Y_METRIC = "rebound_mm"

from bodaqs_analysis.library_api import LibraryAdapter

adapter = LibraryAdapter(LIBRARIES_ROOT)
libraries = {item["library_id"]: item for item in adapter.list_libraries()}
if LIBRARY_ID not in libraries:
    available = ", ".join(sorted(libraries)) or "none found"
    raise ValueError(f"Library {LIBRARY_ID!r} was not found. Available libraries: {available}")

LIBRARY_ROOT = Path(libraries[LIBRARY_ID]["root"])
FALLBACK_EVENT_SCHEMA_PATH = ANALYSIS_DIR / "event schema" / "event_schema.yaml"
pio.renderers.default = "notebook_connected"

print(f"Analysis package root: {ANALYSIS_DIR}")
print(f"Library root: {LIBRARY_ROOT}")


Analysis package root: C:\Users\benco\dev\BODAQS\analysis
Library root: C:\Users\benco\OneDrive\BODAQS-data\archie


In [2]:
from bodaqs_analysis.library_preprocessing import (
    PreprocessBatchRequest,
    batch_result_to_study_set,
    preprocess_requested_sessions_to_library,
)

if not INPUT_PATHS:
    raise ValueError("Add one or more paths to INPUT_PATHS before running this cell.")


def progress(event, payload):
    if event in {"input_started", "input_succeeded", "input_failed"}:
        print(f"{event}: {payload.get('input_path')} {payload.get('session_id', '')} {payload.get('error', '')}")


batch_result = preprocess_requested_sessions_to_library(
    PreprocessBatchRequest(
        artifacts_dir=LIBRARY_ROOT,
        input_paths=tuple(Path(p) for p in INPUT_PATHS),
        preprocess_profile_path=PREPROCESS_PROFILE_PATH,
        bike_profile_path=BIKE_PROFILE_PATH,
        run_tz_label=RUN_TZ_LABEL,
        run_description=RUN_DESCRIPTION,
        generic_log_metadata_paths=tuple(Path(p) for p in GENERIC_LOG_METADATA_PATHS),
        log_metadata_path=LOG_METADATA_PATH,
        fit_dir=FIT_DIR,
        fit_bindings_path=FIT_BINDINGS_PATH,
        logger_timezone=LOGGER_TIMEZONE,
        include_events=True,
        include_metrics=True,
        attach_draft_note=ATTACH_DRAFT_NOTE,
        session_note_template_path=SESSION_NOTE_TEMPLATE_PATH if ATTACH_DRAFT_NOTE else None,
        continue_on_error=True,
    ),
    progress_callback=progress,
)

print(f"Run written: {batch_result['run_id']}")
print(f"Run manifest: {batch_result['run_manifest_path']}")
display(pd.DataFrame(batch_result["results"]))


ValueError: Add one or more paths to INPUT_PATHS before running this cell.

In [ ]:
from bodaqs_analysis.dashboards import make_simple_suspension_metrics_dashboard
from bodaqs_analysis.library_api import make_study_set_selector_handle
from bodaqs_analysis.library_api.selection import study_set_to_selection_snapshot
from bodaqs_analysis.widgets.event_schema_resolution import (
    EventSchemaResolutionError,
    resolve_event_schema_for_selection,
)

study_set = batch_result_to_study_set(batch_result, library_id=LIBRARY_ID)
if not study_set["sessions"]:
    raise ValueError("No sessions were successfully processed, so there is nothing to chart.")

study_set_bridge = study_set_to_selection_snapshot(
    LIBRARY_ROOT,
    study_set,
    include_groupings=False,
)
sel = make_study_set_selector_handle(
    study_set_bridge,
    title="Freshly processed sessions",
    rows=8,
    select_first_by_default=True,
)
display(sel["ui"])

try:
    schema_resolution = resolve_event_schema_for_selection(
        sel,
        fallback_schema_path=FALLBACK_EVENT_SCHEMA_PATH,
    )
except EventSchemaResolutionError as exc:
    raise RuntimeError("Processed sessions do not share one event schema.") from exc

for warning in schema_resolution.warnings:
    print(f"Warning: {warning}")
print(f"Schema source: {schema_resolution.source}")

dashboard = make_simple_suspension_metrics_dashboard(
    sel,
    front_displacement_selector=FRONT_SUSPENSION_SELECTOR,
    rear_displacement_selector=REAR_SUSPENSION_SELECTOR,
    front_velocity_selector=FRONT_SUSPENSION_SELECTOR,
    rear_velocity_selector=REAR_SUSPENSION_SELECTOR,
    front_event_signal_selector=FRONT_EVENT_SIGNAL_SELECTOR,
    rear_event_signal_selector=REAR_EVENT_SIGNAL_SELECTOR,
    compression_event_type=SCATTER_COMPRESSION_EVENT_ID,
    rebound_event_type=SCATTER_REBOUND_EVENT_ID,
    scatter_x_metric=SCATTER_X_METRIC,
    compression_y_metric=SCATTER_COMPRESSION_Y_METRIC,
    rebound_y_metric=SCATTER_REBOUND_Y_METRIC,
)
display(dashboard["ui"])
